In [0]:
from pyspark.sql import functions as F

df = spark.table("workspace.home_credit_bronze.application_train")

# 1. literal 'null' strings
df.filter(F.col("OWN_CAR_AGE") == "null").count()

# 2. DAYS_EMPLOYED sentinel
df.groupBy("DAYS_EMPLOYED").count().orderBy(F.desc("count")).show(5)

# 3. unknown-value tokens
df.groupBy("CODE_GENDER").count().show()
df.groupBy("ORGANIZATION_TYPE").count().orderBy(F.desc("count")).show(5, truncate=False)

# 4. flag values
df.groupBy("FLAG_OWN_CAR", "FLAG_MOBIL").count().show()

# 5. EXT_SOURCE_2 range
df.select(
    F.min(F.expr("try_cast(EXT_SOURCE_2 AS DOUBLE)")),
    F.max(F.expr("try_cast(EXT_SOURCE_2 AS DOUBLE)")),
).show()

In [0]:
df.filter((F.col("DAYS_EMPLOYED") == "365243") & (F.col("ORGANIZATION_TYPE") == "XNA")).count()

In [0]:
df.filter(F.col("OWN_CAR_AGE") == "null").count()
df.filter(F.col("DAYS_EMPLOYED") == "365243") \
  .groupBy("NAME_INCOME_TYPE").count().orderBy(F.desc("count")).show()

In [0]:
df = spark.table("workspace.home_credit_bronze.bureau")
df.limit(5).display()

In [0]:
b = spark.table("workspace.home_credit_bronze.bureau")
a = spark.table("workspace.home_credit_bronze.application_train")

b.groupBy("CREDIT_ACTIVE").count().show()
b.groupBy("CREDIT_CURRENCY").count().show()
b.groupBy("CREDIT_TYPE").count().orderBy(F.desc("count")).show(truncate=False)

# key uniqueness
print(b.count(), b.select("SK_ID_BUREAU").distinct().count())

# orphans: bureau customers missing from application_train
b.select("SK_ID_CURR").distinct().join(a.select("SK_ID_CURR"), "SK_ID_CURR", "left_anti").count()

In [0]:
%sql
SELECT COUNT(*) FROM workspace.home_credit_silver.customer UNION ALL            -- expect 307,511
SELECT COUNT(*) FROM workspace.home_credit_silver.customer_quarantine UNION ALL  -- expect 0
SELECT COUNT(*) - COUNT(DISTINCT sk_id_curr) AS duplicate_keys
FROM workspace.home_credit_silver.customer;    

In [0]:
s = spark.table("workspace.home_credit_silver.customer")

# sentinel gone, is_employed correct (expect 55,374 false)
s.filter("is_employed = false").count()
s.filter("days_employed = 365243").count()          # expect 0

# XNA gone (expect 0 and 0)
s.filter("code_gender = 'XNA'").count()
s.filter("organization_type = 'XNA'").count()

# types and lowercase names look right
s.printSchema()
s.select("sk_id_curr", "age_years", "years_employed", "is_employed", "flag_own_car").show(5)

In [0]:
df=spark.table("workspace.home_credit_bronze.bureau")

df.limit(5).display()

In [0]:
from pyspark.sql import functions as F


b = spark.table("workspace.home_credit_bronze.bureau")
b.groupBy("CREDIT_ACTIVE").agg(
    F.count("*").alias("n"),
    F.sum(F.col("DAYS_CREDIT_ENDDATE").isNull().cast("int")).alias("null_enddate"),
    F.sum((F.col("DAYS_ENDDATE_FACT").isNull()).cast("int")).alias("null_enddate_fact"),
).show()



In [0]:
for c in ["DAYS_CREDIT", "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE"]:
    print(c)
    b.groupBy(c).count().orderBy(F.desc("count")).show(3)

In [0]:
b.select(
    F.min(F.expr("try_cast(DAYS_CREDIT_ENDDATE AS DOUBLE)")).alias("min_enddate"),
    F.max(F.expr("try_cast(DAYS_CREDIT_ENDDATE AS DOUBLE)")).alias("max_enddate"),
    F.min(F.expr("try_cast(DAYS_ENDDATE_FACT AS DOUBLE)")).alias("min_fact"),
    F.max(F.expr("try_cast(DAYS_ENDDATE_FACT AS DOUBLE)")).alias("max_fact"),
    F.min(F.expr("try_cast(AMT_CREDIT_SUM AS DOUBLE)")).alias("min_sum"),
    F.max(F.expr("try_cast(AMT_CREDIT_SUM AS DOUBLE)")).alias("max_sum"),
).show()

In [0]:
b.select(
    F.sum((F.abs(F.expr("try_cast(DAYS_CREDIT_ENDDATE AS DOUBLE)")) > 36500).cast("int")).alias("enddate_over_100y"),
    F.sum((F.abs(F.expr("try_cast(DAYS_ENDDATE_FACT AS DOUBLE)")) > 36500).cast("int")).alias("fact_over_100y"),
    F.sum((F.expr("try_cast(AMT_CREDIT_SUM AS DOUBLE)") > 100000000).cast("int")).alias("sum_over_100m"),
).show()

In [0]:
from pyspark.sql import functions as F  
b = spark.table("workspace.home_credit_bronze.bureau")
b.filter(F.expr("try_cast(DAYS_CREDIT_ENDDATE AS DOUBLE) < try_cast(DAYS_CREDIT AS DOUBLE)")).count()
# b.filter(F.expr("try_cast(DAYS_ENDDATE_FACT AS DOUBLE) < try_cast(DAYS_CREDIT AS DOUBLE)")).count()

In [0]:
%sql
SELECT COUNT(*) FROM workspace.home_credit_silver.bureau_loan WHERE end_date_nulled UNION ALL
SELECT COUNT(*) FROM workspace.home_credit_silver.bureau_loan WHERE status_date_conflict;  -- expect 1,968

In [0]:
b=spark.table("workspace.home_credit_bronze.credit_card_balance")
b.limit(5).display()

In [0]:
cc = spark.table("workspace.home_credit_bronze.credit_card_balance")
cc.groupBy("NAME_CONTRACT_STATUS").count().show()                       # expect ~7 clean statuses
cc.select(F.min(F.col("MONTHS_BALANCE").cast("int")), F.max(F.col("MONTHS_BALANCE").cast("int"))).show()  # expect about -96 to -1
cc.select(F.min(F.expr("try_cast(SK_DPD AS INT)"))).show()              # expect 0 (a negative would be odd)
cc.select(F.sum((F.expr("try_cast(AMT_BALANCE AS DOUBLE)") < 0).cast("int")),
          F.sum((F.expr("try_cast(AMT_PAYMENT_CURRENT AS DOUBLE)") < 0).cast("int"))).show()   # negatives
# orphans: customers and previous applications that don't exist upstream
a = spark.table("workspace.home_credit_bronze.application_train")
p = spark.table("workspace.home_credit_bronze.previous_application")
cc.select("SK_ID_CURR").distinct().join(a.select("SK_ID_CURR"), "SK_ID_CURR", "left_anti").count()
cc.select("SK_ID_PREV").distinct().join(p.select("SK_ID_PREV"), "SK_ID_PREV", "left_anti").count()

In [0]:
cc.select("SK_ID_CURR").distinct().join(a.select("SK_ID_CURR"), "SK_ID_CURR", "left_anti").count()

In [0]:
cc.select("SK_ID_PREV").distinct().join(p.select("SK_ID_PREV"), "SK_ID_PREV", "left_anti").count()

In [0]:
%sql
SELECT COUNT(*) FROM workspace.home_credit_silver.credit_card_balance;             -- 3,840,312
SELECT COUNT(*) FROM workspace.home_credit_silver.credit_card_balance_quarantine;  -- 0

In [0]:
b=spark.table("workspace.home_credit_bronze.previous_application")
b.limit(5).display()

In [0]:
from pyspark.sql import functions as F  
p = spark.table("workspace.home_credit_bronze.previous_application")

print(p.count(), p.select("SK_ID_PREV").distinct().count())     # key unique?
p.groupBy("NAME_CONTRACT_STATUS").count().show()                # expect 4 values
p.groupBy("CODE_REJECT_REASON").count().show()

for c in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION",
          "DAYS_LAST_DUE", "DAYS_TERMINATION", "SELLERPLACE_AREA"]:
    print(c); p.groupBy(c).count().orderBy(F.desc("count")).show()

for c in ["NAME_PAYMENT_TYPE", "NAME_CASH_LOAN_PURPOSE", "NAME_GOODS_CATEGORY",
          "NAME_SELLER_INDUSTRY", "NAME_PRODUCT_TYPE", "NAME_YIELD_GROUP"]:
    print(c, p.filter(F.col(c) == "XNA").count())

# orphans: customers missing from application_train
a = spark.table("workspace.home_credit_bronze.application_train")
p.select("SK_ID_CURR").distinct().join(a.select("SK_ID_CURR"), "SK_ID_CURR", "left_anti").count()

In [0]:
%sql
SELECT COUNT(*) FROM workspace.home_credit_silver.previous_application;             -- 1,670,214
SELECT COUNT(*) FROM workspace.home_credit_silver.previous_application_quarantine;  -- 0
SELECT COUNT(*) FROM workspace.home_credit_silver.previous_application
WHERE days_first_drawing IS NULL;                                                   -- 1,607,509
SELECT COUNT(*) FROM workspace.home_credit_silver.previous_application
WHERE sellerplace_area = -1;                                                        -- 762,675 